# Labwork 2

Run the following cell first. It reports what hardware you were given and stops the notebook if that is a CPU: the runtime is declared in this file, but should you land on a CPU anyway, go to **Runtime ▸ Change runtime type ▸ T4 GPU** and run it again.

In [22]:
!nvidia-smi
!nvcc --version

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("numba") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"],
                   check=True)

import numba

try:
    # Recent numba keeps the CUDA target in a separate distribution; the one
    # Colab ships still has it built in. Ask, rather than assume either way.
    from numba import config, cuda
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba-cuda"],
                   check=True)
    from numba import config, cuda

config.CUDA_ENABLE_PYNVJITLINK = True

print(f"numba {numba.__version__}")

if not cuda.is_available():
    raise SystemExit(
        "No CUDA device: Runtime -> Change runtime type -> T4 GPU, "
        "then Runtime -> Run all. Nothing below this cell can run on a CPU."
    )
cuda.detect()

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvcc: command not found
numba 0.67.0
Found 1 CUDA devices
id 0    NVIDIA GeForce RTX 5060 Ti                              [SUPPORTED]
                      Compute Capability: 12.0
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-fd7a79a8-dba9-c363-20c2-85cfb7c05ea9
                                Watchdog: Enabled
             FP32/FP64 Performance Ratio: 64
                            FP16 Support: Yes
                            BF16 Support: Yes
                             FP8 Support: Yes
Summary:
	1/1 devices are supported


True

### Checking your work

The cell below defines two helpers, provided: there is nothing to fill in.

`verify(...)` compares your result with a NumPy reference and **raises** when
they disagree, naming the first index that differs. A red cell means a wrong
kernel, a green one a right one.

`benchmark(...)` times the GPU against NumPy over a range of sizes, after
warm-up runs that pay for the JIT compilation: timing the first launch measures
the compiler, not the kernel.

Each exercise is followed by its own **`testN`**, which runs your class and
hands the result to the oracle for exercise *N*. You may call `test1(...)`
again at any point, on any array you like, and it still tests exercise 1.

In [23]:
# --- provided: the self-check and benchmark helpers --------------------------
import contextlib
import io
from time import perf_counter

import numpy as np
from numba import core, cuda

# The first exercises launch a single block on purpose, and numba warns about
# it every time. The warning is right in general and noise here.
core.config.CUDA_LOW_OCCUPANCY_WARNINGS = False


class CheckFailed(AssertionError):
    """Raised when a kernel disagrees with its CPU reference."""


# Named `verify` rather than `check` because each exercise gets its own
# `checkN`: that one is the oracle for exercise N, this one is the primitive
# they are all written in terms of.


def verify(name, got, expected, tolerance=0.0):
    """Compare a result against its oracle, and raise on the first difference.

    `tolerance` is relative; leave it at 0 for integers, where the answer is
    exact, and pass something like 1e-5 for floats.
    """
    got, expected = np.asarray(got), np.asarray(expected)
    if got.shape != expected.shape:
        raise CheckFailed(
            f"{name}: shape is {got.shape}, expected {expected.shape}"
        )
    if tolerance:
        wrong = ~np.isclose(got, expected, rtol=tolerance, atol=tolerance)
    else:
        wrong = got != expected
    count = int(np.count_nonzero(wrong))
    if count:
        first = int(np.argmax(wrong))
        raise CheckFailed(
            f"{name}: {count} of {got.size} values are wrong. "
            f"First at index {first}: got {got.ravel()[first]}, "
            f"expected {expected.ravel()[first]}."
        )
    print(f"PASS  {name}  ({got.size} value{'' if got.size == 1 else 's'})")


def _time_once(call):
    """One timed call, in milliseconds, with the device drained first.

    Most of these classes print their own launch sizes and timings on every
    call, which would bury the table below under a few hundred lines, so the
    timed call is run with its output swallowed.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        start = perf_counter()
        call()
        cuda.synchronize()
        stop = perf_counter()
    return (stop - start) * 1000.0


def peak_bandwidth():
    """The card's theoretical memory bandwidth in GB/s, or None if unreadable.

    Peak is what a reduction or a scan should be measured against. These
    kernels do almost no arithmetic: they read the array once and produce a
    handful of numbers, so the only question worth asking is how close to the
    memory system they get. A speed-up against one CPU core answers a much
    easier question.
    """
    try:
        device = cuda.get_current_device()
        clock_khz = device.MEMORY_CLOCK_RATE            # kHz
        bus_bytes = device.GLOBAL_MEMORY_BUS_WIDTH / 8  # bits -> bytes
        return 2 * clock_khz * 1e3 * bus_bytes / 1e9    # 2: DDR
    except Exception:
        return None


def benchmark(name, build, sizes, repeats=10, warmup=3):
    """Time the GPU against NumPy over `sizes`.

    `build(size)` returns `(gpu, cpu)`, or `(gpu, cpu, nbytes)` to get the
    bandwidth columns as well. The two callables take no arguments and already
    hold their data, so the upload is not timed unless the kernel needs it.
    `nbytes` is the traffic one call claims to move; what counts as traffic is
    said where each `build` is written, because it is not the same question for
    a reduction, a scan and a sort.
    """
    peak = peak_bandwidth()
    print(name + (f"   —   this card peaks at {peak:.0f} GB/s" if peak else ""))
    print(f"{'size':>12} | {'GPU (ms)':>9} | {'NumPy (ms)':>11} | "
          f"{'speed-up':>9} | {'GB/s':>7} | {'of peak':>8}")
    print("-" * 72)
    for size in sizes:
        with contextlib.redirect_stdout(io.StringIO()):
            built = build(size)
            gpu, cpu = built[0], built[1]
            nbytes = built[2] if len(built) > 2 else None
            for _ in range(warmup):
                gpu()
            cuda.synchronize()
        best = min(_time_once(gpu) for _ in range(repeats))
        reference = min(_time_once(cpu) for _ in range(3))

        rate = None if nbytes is None else nbytes / (best / 1000.0) / 1e9
        rate_text = "      -" if rate is None else f"{rate:7.1f}"
        share_text = ("       -" if rate is None or peak is None
                      else f"{100 * rate / peak:7.1f}%")
        print(f"{size:12d} | {best:9.3f} | {reference:11.3f} | "
              f"{reference / best:8.1f}x | {rate_text} | {share_text}")

Run the following cell to verify that numba can use Cuda...

In [24]:
%%python
from numba import cuda

def bool_to_str(predicate: bool) -> str:
    if predicate:
        return 'yes'
    return 'no'

def is_cuda_available_str() -> str:
    return bool_to_str(cuda.is_available())

print(f'is cuda available? {is_cuda_available_str()}')

is cuda available? yes


Another simple solution to detect and print the devices from [numba documentation](https://numba.readthedocs.io/en/stable/cuda-reference/host.html#device-management):

In [25]:
%%python
from numba import cuda

if not cuda.detect():
    raise Exception("we do not have cuda :-(")

Found 1 CUDA devices
id 0    NVIDIA GeForce RTX 5060 Ti                              [SUPPORTED]
                      Compute Capability: 12.0
                           PCI Device ID: 0
                              PCI Bus ID: 1
                                    UUID: GPU-fd7a79a8-dba9-c363-20c2-85cfb7c05ea9
                                Watchdog: Enabled
             FP32/FP64 Performance Ratio: 64
                            FP16 Support: Yes
                            BF16 Support: Yes
                             FP8 Support: Yes
Summary:
	1/1 devices are supported


Run the following cell for a short test of numba/cuda...

In [26]:
%%python
# This is a MAP example
from numba import cuda
import numpy as np 


@cuda.jit
def increment_by_one(an_array):
    # Thread id in a 1D block
    tx = cuda.threadIdx.x
    # Block id in a 1D grid
    ty = cuda.blockIdx.x
    # Block width, i.e. number of threads per block
    bw = cuda.blockDim.x
    # Compute flattened index inside the array
    pos = tx + ty * bw
    if pos < an_array.size:  # Check array boundaries
        an_array[pos] += 1
        

if __name__ == "__main__":
    # build a big vector
    h_array = np.arange(1<<16)
    
    d_array = cuda.to_device(h_array)
    
    threads_per_block = 32*8 # 8 warps, 256 threads per block
    blocks_per_grid = (h_array.size + (threads_per_block - 1)) // threads_per_block
    increment_by_one[blocks_per_grid, threads_per_block](d_array)
    
    d_array.copy_to_host(h_array)
    
    for i in range(len(h_array)):
        assert h_array[i] == i+1, f'bad value at index {i}: {h_array[i]+1}'
    print('it woks!')

it woks!


From here, it is clear that some code are easy to do:
- binary transform
- gather
- scatter
- and basic reduce

The first labwork (of week 2) introduces a quite simple but not generic reduce. 
This version suffers from many default:
- the operation is hard-coded ;
- this version is not optimal considering data loading...

The last point needs some explanations. Actually, when a thread loads one data, it is through a vector-instruction and implies the collaboration of the 32 threads of the same warp. 
If all the threads of a warp needs to load data, then we have different cases:
1. The threads are loading **coalescent** data (contiguous, located into a 128 bytes array or less).
2. The threads are not loading **coalescent** data...

For the first case, actually the 32 loads are managed together through a single vector operation. 
For the second case, then the SIMD processor needs (up to) 32 data load instructions. 

Obviously the first case **is** much more fast than the second! Especially when you consider the data loading processing time, which is quite huge (some hundreds of cycles!). It is very important to try to load all the data in the first configuration with coalescent loading. The same for the writing. 

Then, in this lab work you will revisit the REDUCE pattern, considering different optimizations regarding the data loading and the operator properties (associativity, but also commutativity).

## First exercise
In this first version, we restart with the more general REDUCE version given in the lectures 1 and 2.

Here, the operator is associative but not commutative (for instance, matrix multiplication is such an operator!). 

Take a look a the implementation, especially to the kernel...

Complete the kernel where needed only (TODO keyword).

In [27]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core, config


class BasicReduce(object):
    """Create a reduction object that reduces values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 32

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the reduction problem...

        This function returns a Cuda Kernel that does the reduction of some data using a given binary functor."""

        reduce_op = cuda.jit(device=True)(fn)

        max_block_size = BasicReduce._NUM_WARPS * BasicReduce._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, arr, null_value):
            global_tid = cuda.grid(1)
            local_tid = cuda.threadIdx.x
            # TODO: fill shared_memory
            shared_memory[local_tid] = arr[global_tid] if global_tid < arr.size else null_value

            # wait all warps
            cuda.syncthreads()

        @cuda.jit(device=True)
        def pointer_jumping(shared_memory, jump):
            local_tid = cuda.threadIdx.x
            right = local_tid + jump
            # TODO: implement the pointer jumping with synchronization
            # NB: it is a general reduce, the binary operation is "reduce_op(l,r)"

            # Correction by professor
            active = right < cuda.blockDim.x

            temp = shared_memory[right if active else local_tid]
            cuda.syncthreads()
            if active:
                shared_memory[local_tid] = reduce_op(shared_memory[local_tid], temp)
            cuda.syncthreads()

        def gpu_reduce_block(arr, partials, null_value):
            """
            Per block reduction, basic version...
            """
            # move data to shared memory
            shared_memory = cuda.shared.array(shape=max_block_size, dtype=np_type)
            load_shared_memory(shared_memory, arr, null_value)

            # do the jumping...
            blk_size = cuda.blockDim.x
            jump = 1
            while jump < blk_size:
                pointer_jumping(shared_memory, jump)
                jump = jump * 2

            # now stores the result
            if cuda.threadIdx.x == 0:
                blk_id = cuda.blockIdx.x
                partials[blk_id] = shared_memory[0]

        return cuda.jit(gpu_reduce_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        reduction. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = BasicReduce._gpu_kernel_factory(self._functor, from_dtype(dtype))
        return self._cache[key]

    def __call__(self, arr, null_value, res=None, stream=cuda.default_stream()):
        """Performs a full reduction.

        :param arr: A host or device array.
        :param null_value: zero value for array.dtype
        :param res: Device array into which to write the reduction
                    result to. The result is written into the first element of
                    this array if this array is provided.
        :param stream: Optional CUDA stream in which to perform the reduction.
                    If no stream is specified, the default stream of 0 is
                    used.
        """

        # ensure 1d array
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if arr.size < 1:
            raise ValueError("array's length is 0")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel = self._compile(arr.dtype)
        nb_threads = self._NUM_WARPS * self._WARP_SIZE
        if True:
            # turn on google colaboratory
            if nb_threads > arr.size:
                nb_threads = arr.size
            nb_blocks = (arr.size + nb_threads - 1) // nb_threads
            temp = cuda.device_array(shape=nb_blocks, dtype=arr.dtype, stream=stream)
            kernel[nb_blocks, nb_threads, stream](arr, temp, null_value)

        # Perform the reduction on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        while True:
            if nb_threads > arr.size:
                nb_threads = arr.size

            nb_blocks = (arr.size + nb_threads - 1) // nb_threads

            temp = cuda.device_array(shape=nb_blocks, dtype=arr.dtype, stream=stream)

            kernel[nb_blocks, nb_threads, stream](arr, temp, null_value)
            print(f"launch {nb_blocks} for {nb_threads * nb_blocks} threads")
            cuda.synchronize()
            arr = temp
            if nb_blocks == 1:
                break

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        # handle return value
        if res is not None:
            res.copy_to_device(arr)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # Returned, not printed: the caller decides what the timing is
        # for. A benchmark keeps the number, a self-check ignores it,
        # and neither has to scrape it back out of the console.
        return arr.copy_to_host(stream=stream)[0], ct


#### Tests — exercise 1 — the general reduce

In [28]:
def test1(h_input, reduce_class=BasicReduce):
    """Upload, reduce, check; return the kernel time the class measured.

    `reduce_class` is captured at definition time, as in every `testN` here, so
    all four stay callable to the end of the notebook whatever comes after
    them. Each exercise also gives its class a name of its own, so the four
    never collide in the first place.
    """
    def check1(result):
        """The sum has to be exact.

        The values are drawn small enough that the true sum still fits in the
        array's own dtype, so this is an equality and not a tolerance.
        """
        verify(f"sum of {h_input.size} values", result,
               h_input.dtype.type(h_input.sum()))

    reducer = reduce_class(lambda a, b: a + b)
    result, ct = reducer(cuda.to_device(h_input), h_input.dtype.type(0))
    check1(result)
    return ct


test1(np.ones(1 << 16, dtype=np.int32))

# Random values rather than ones: a kernel that drops or double-counts an
# element gives the right answer on a constant array surprisingly often.
rng = np.random.default_rng(1)
test1(rng.integers(0, 1 << 10, size=1 << 16, dtype=np.int32))

# The closing section of this notebook compares all four reduces, which it can
# only do if all four classes are still reachable by then.
ReduceExercise1 = BasicReduce

launch 64 for 65536 threads
launch 1 for 64 threads
PASS  sum of 65536 values  (1 value)
launch 64 for 65536 threads
launch 1 for 64 threads
PASS  sum of 65536 values  (1 value)


## Second exercise
One big problem with the previous implementation is linked to the **register pressure**. 

A block of threads runs onto a SMP, with others blocks. 
But a SMP has a limited number of registers, to partition to all the threads running into blocks. 

When a program uses too much threads, then the number of blocks running onto a single SMP is reduced... That is a limit to the speed-up since, at the same time, using less blocks implies less threads and so longer computation time. 

Using shared memory impacts directly on the number of registers, since the shared memory uses SMP registers... It is then important to reduce the size of the shared memory, and so in our reduce algorithm the size of the blocks. 

Try the same code but with 256 **threads** only. Notice the computation times...

In [29]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core, config


class SmallBlockReduce(object):
    """Create a reduction object that reduces values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 4 # TODO: modify this value...

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the reduction problem...

        This function returns a Cuda Kernel that does the reduction of some data using a given binary functor."""

        reduce_op = cuda.jit(device=True)(fn)

        max_block_size = SmallBlockReduce._NUM_WARPS * SmallBlockReduce._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, arr, null_value):
            global_tid = cuda.grid(1)
            local_tid = cuda.threadIdx.x
            # TODO: fill shared_memory
            shared_memory[local_tid] = arr[global_tid] if global_tid < arr.size else null_value
                        
            # wait all warps
            cuda.syncthreads()

        @cuda.jit(device=True)
        def pointer_jumping(shared_memory, jump):
            local_tid = cuda.threadIdx.x
            right = local_tid + jump
            # TODO: implement the pointer jumping with synchronization
            active = right < cuda.blockDim.x
            
            temp = shared_memory[right if active else local_tid]
            cuda.syncthreads()
            if active:
                shared_memory[local_tid] = reduce_op(shared_memory[local_tid], temp)
            cuda.syncthreads()


        def gpu_reduce_block(arr, partials, null_value):
            """
            Per block reduction, basic version...
            """
            # move data to shared memory
            shared_memory = cuda.shared.array(shape=max_block_size, dtype=np_type)
            load_shared_memory(shared_memory, arr, null_value)

            # do the jumping...
            blk_size = cuda.blockDim.x
            jump = 1
            while jump < blk_size:
                pointer_jumping(shared_memory, jump)
                jump = jump * 2

            # now stores the result
            if cuda.threadIdx.x == 0:
                blk_id = cuda.blockIdx.x
                partials[blk_id] = shared_memory[0]

        return cuda.jit(gpu_reduce_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        reduction. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = SmallBlockReduce._gpu_kernel_factory(self._functor, from_dtype(dtype))
        return self._cache[key]

    def __call__(self, arr, null_value, res=None, stream=cuda.default_stream()):
        """Performs a full reduction.

        :param arr: A host or device array.
        :param null_value: zero value for array.dtype
        :param res: Device array into which to write the reduction
                    result to. The result is written into the first element of
                    this array if this array is provided.
        :param stream: Optional CUDA stream in which to perform the reduction.
                    If no stream is specified, the default stream of 0 is
                    used.
        """

        # ensure 1d array
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if arr.size < 1:
            raise ValueError("array's length is 0")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel = self._compile(arr.dtype)
        nb_threads = self._NUM_WARPS * self._WARP_SIZE
        if True:
            # turn on google Colaboratory
            if nb_threads > arr.size:
                nb_threads = arr.size
            nb_blocks = (arr.size + nb_threads - 1) // nb_threads
            temp = cuda.device_array(shape=nb_blocks, dtype=arr.dtype, stream=stream)
            kernel[nb_blocks, nb_threads, stream](arr, temp, null_value)

        # Perform the reduction on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        while True:
            if nb_threads > arr.size:
                nb_threads = arr.size

            nb_blocks = (arr.size + nb_threads - 1) // nb_threads

            temp = cuda.device_array(shape=nb_blocks, dtype=arr.dtype, stream=stream)

            kernel[nb_blocks, nb_threads, stream](arr, temp, null_value)
            print(f"launch {nb_blocks} for {nb_threads * nb_blocks} threads")
            cuda.synchronize()
            arr = temp
            if nb_blocks == 1:
                break

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        # handle return value
        if res is not None:
            res.copy_to_device(arr)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # Returned, not printed: the caller decides what the timing is
        # for. A benchmark keeps the number, a self-check ignores it,
        # and neither has to scrape it back out of the console.
        return arr.copy_to_host(stream=stream)[0], ct


#### Tests — exercise 2 — the same reduce, with less shared memory

In [30]:
def test2(h_input, reduce_class=SmallBlockReduce):
    def check2(result):
        verify(f"sum of {h_input.size} values", result,
               h_input.dtype.type(h_input.sum()))

    reducer = reduce_class(lambda a, b: a + b)
    result, ct = reducer(cuda.to_device(h_input), h_input.dtype.type(0))
    check2(result)
    return ct


rng = np.random.default_rng(2)
for size in (1 << 12, 1 << 16, (1 << 18) + 101):  # the last is not a power of 2
    test2(rng.integers(0, 1 << 10, size=size, dtype=np.int32))

ReduceExercise2 = SmallBlockReduce

launch 32 for 4096 threads
launch 1 for 32 threads
PASS  sum of 4096 values  (1 value)
launch 512 for 65536 threads
launch 4 for 512 threads
launch 1 for 4 threads
PASS  sum of 65536 values  (1 value)
launch 2049 for 262272 threads
launch 17 for 2176 threads
launch 1 for 17 threads
PASS  sum of 262245 values  (1 value)


## Third exercise
The objective here is to optimize the former reduce.
We considerer here that the binary operation is associative but not *commutative*.
So, we cannot permute the data...

The simpler optimization consists to do:
- a reduce per warp,
- write the per warp result at position warp_id (so `threadIdx.x // 32`),
- conclude by a last reduce per warp for warp 0,
- and save the per block result. 

Implement this strategy and note the speed-up...

In [31]:

from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core, config


class AssociativeReduce(object):
    """Create a reduction object that reduces values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the reduction problem."""

        reduce_op = cuda.jit(device=True)(fn)

        max_block_size = (
            AssociativeReduce._NUM_WARPS
            * AssociativeReduce._WARP_SIZE
        )

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, arr, null_value):
            global_tid = cuda.grid(1)
            local_tid = cuda.threadIdx.x

            shared_memory[local_tid] = (
                arr[global_tid]
                if global_tid < arr.size
                else null_value
            )

            cuda.syncthreads()

        @cuda.jit(device=True)
        def reduce_per_warp(shared_memory):
            tid = cuda.threadIdx.x
            lane = tid & 31

            jump = 1
            while jump < 32:
                active = lane + jump < 32

                if active:
                    temp = shared_memory[tid + jump]
                else:
                    temp = shared_memory[tid]

                # All threads must reach each block-wide barrier.
                cuda.syncthreads()

                if active:
                    shared_memory[tid] = reduce_op(
                        shared_memory[tid], temp
                    )

                cuda.syncthreads()
                jump *= 2

        def gpu_reduce_block(arr, partials, null_value):
            shared_memory = cuda.shared.array(
                shape=max_block_size,
                dtype=np_type
            )

            tid = cuda.threadIdx.x
            num_warps = (cuda.blockDim.x + 31) // 32

            # Step 1: Load input into shared memory.
            load_shared_memory(
                shared_memory, arr, null_value
            )

            # Step 2: Reduce each warp's segment.
            reduce_per_warp(shared_memory)
            cuda.syncthreads()

            # Step 3: Collect the result of each warp.
            if tid < 32:
                if tid < num_warps:
                    shared_memory[tid] = shared_memory[tid * 32]
                else:
                    shared_memory[tid] = null_value

            cuda.syncthreads()

            # Step 4: Reduce the collected warp results.
            reduce_per_warp(shared_memory)
            cuda.syncthreads()

            # Step 5: Write one partial result per block.
            if tid == 0:
                partials[cuda.blockIdx.x] = shared_memory[0]

        return cuda.jit(gpu_reduce_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        reduction. It will be compiled as a CUDA device
                        function using cuda.jit(device=True).
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = AssociativeReduce._gpu_kernel_factory(
                self._functor, from_dtype(dtype)
            )
        return self._cache[key]

    def __call__(
        self,
        arr,
        null_value,
        res=None,
        stream=cuda.default_stream()
    ):
        """Performs a full reduction."""

        # Ensure 1D array.
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # Ensure size > 0.
        if arr.size < 1:
            raise ValueError("array's length is 0")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel = self._compile(arr.dtype)
        nb_threads = self._NUM_WARPS * self._WARP_SIZE

        # Perform the reduction on the GPU.
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        while True:
            nb_blocks = (arr.size + nb_threads - 1) // nb_threads

            temp = cuda.device_array(
                shape=nb_blocks,
                dtype=arr.dtype,
                stream=stream
            )

            kernel[nb_blocks, nb_threads, stream](
                arr, temp, null_value
            )

            arr = temp

            if nb_blocks == 1:
                break

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()

        ct = cuda.event_elapsed_time(start_event, stop_event)

        # Handle return value.
        if res is not None:
            res.copy_to_device(arr, stream=stream)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        return arr.copy_to_host(stream=stream)[0], ct


def test3(h_input, reduce_class=AssociativeReduce):
    def check3(result):
        verify(
            f"sum of {h_input.size} values",
            result,
            h_input.dtype.type(h_input.sum())
        )

    reducer = reduce_class(lambda a, b: a + b)
    result, ct = reducer(
        cuda.to_device(h_input),
        h_input.dtype.type(0)
    )
    check3(result)
    return ct


rng = np.random.default_rng(3)

test3(
    rng.integers(
        0, 1 << 10,
        size=1 << 18,
        dtype=np.int32
    )
)

test3(np.ones(1 << 18, dtype=np.float32))

ReduceExercise3 = AssociativeReduce


PASS  sum of 262144 values  (1 value)
PASS  sum of 262144 values  (1 value)


#### Tests — exercise 3 — one reduce per warp

In [32]:
def test3(h_input, reduce_class=AssociativeReduce):
    def check3(result):
        verify(f"sum of {h_input.size} values", result,
               h_input.dtype.type(h_input.sum()))

    reducer = reduce_class(lambda a, b: a + b)
    result, ct = reducer(cuda.to_device(h_input), h_input.dtype.type(0))
    check3(result)
    return ct


rng = np.random.default_rng(3)
test3(rng.integers(0, 1 << 10, size=1 << 18, dtype=np.int32))

# Float32 as well. The sum of 2^18 ones is exactly representable, so even in
# floating point this stays an equality rather than a tolerance.
test3(np.ones(1 << 18, dtype=np.float32))

ReduceExercise3 = AssociativeReduce

PASS  sum of 262144 values  (1 value)
PASS  sum of 262144 values  (1 value)


#### Benchmark — one reduce per warp

In [33]:
def build(size):
    h_input = np.random.default_rng(size).integers(
        0, 1 << 10, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    reducer = AssociativeReduce(lambda a, b: a + b)
    zero = np.int32(0)
    # The array is uploaded once, here, and only the reduction is timed: the
    # class reads its input without destroying it, so there is nothing to
    # restore between calls.
    return ((lambda: reducer(d_input, zero)), (lambda: h_input.sum()),
            h_input.nbytes)


benchmark("associative reduce: GPU against numpy.sum",
          build, (1 << 20, 1 << 22, 1 << 24))

associative reduce: GPU against numpy.sum   —   this card peaks at 448 GB/s
        size |  GPU (ms) |  NumPy (ms) |  speed-up |    GB/s |  of peak
------------------------------------------------------------------------
     1048576 |     0.546 |       0.309 |      0.6x |     7.7 |     1.7%


     4194304 |     0.364 |       1.371 |      3.8x |    46.1 |    10.3%
    16777216 |     0.660 |       4.803 |      7.3x |   101.7 |    22.7%


## Fourth exercise
Now, let us consider what happens with a *commutative* binary operation. 
In such a case, it is possible to reduce the number of Cuda blocks to *the number of threads per block* at most, and so to limit the number of recursive calls (at most 2 calls, the last with a single block). 

For this, each thread should:
- do a local reduce in a private register with a modulo strategy, 
- and then terminate the implementation as in the previous exercise. 

Implements this strategy in the following cell. 

In [34]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core, config


class CommutativeReduce(object):
    """Create a reduction object that reduces values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the reduction problem...

        This function returns a Cuda Kernel that does the reduction of some data using a given binary functor."""

        reduce_op = cuda.jit(device=True)(fn)

        max_block_size = CommutativeReduce._NUM_WARPS * CommutativeReduce._WARP_SIZE

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, arr, null_value):
            global_tid = cuda.grid(1)
            local_tid = cuda.threadIdx.x
            # TODO: load with a modulo scheme
            shared_memory[local_tid] = arr[global_tid] if global_tid < arr.size else null_value

            # wait all warps
            cuda.syncthreads()

        @cuda.jit(device=True)
        def reduce_per_warp(shared_memory):
            warp_id = cuda.threadIdx.x // 32
            in_warp_id = cuda.threadIdx.x % 32
            # TODO: implement the pointer jumping without synchronization

            jump = 1
            while jump < 32:
                if in_warp_id % (2 * jump) == 0:
                    shared_memory[tid] = reduce_op(
                        shared_memory[tid],
                        shared_memory[tid + jump]
                    )
                jump *= 2

        def gpu_reduce_block(arr, partials, null_value):
            """
            Per block reduction, basic version...
            """
            # move data to shared memory
            shared_memory = cuda.shared.array(shape=max_block_size, dtype=np_type)
            load_shared_memory(shared_memory, arr, null_value)

            # TODO: implements the logics
            reduce_per_warp(shared_memory)
            cuda.syncthreads()

            jump = 1
            while jump < 8:
                tid = cuda.threadIdx.x

                if tid < 8 and tid % (2 * jump) == 0:
                    shared_memory[tid * 32] = reduce_op(
                        shared_memory[tid * 32],
                        shared_memory[(tid + jump) * 32]
                    )

                cuda.syncthreads()
                jump *= 2

            # now stores the result
            if cuda.threadIdx.x == 0:
                blk_id = cuda.blockIdx.x
                partials[blk_id] = shared_memory[0]

        return cuda.jit(gpu_reduce_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        reduction. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = CommutativeReduce._gpu_kernel_factory(self._functor, from_dtype(dtype))
        return self._cache[key]

    def __call__(self, arr, null_value, res=None, stream=cuda.default_stream()):
        """Performs a full reduction.

        :param arr: A host or device array.
        :param null_value: zero value for array.dtype
        :param res: Device array into which to write the reduction
                    result to. The result is written into the first element of
                    this array if this array is provided.
        :param stream: Optional CUDA stream in which to perform the reduction.
                    If no stream is specified, the default stream of 0 is
                    used.
        """

        # ensure 1d array
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # ensure size > 0
        if arr.size < 1:
            raise ValueError("array's length is 0")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = core.config.CUDA_LOW_OCCUPANCY_WARNINGS, False

        kernel = self._compile(arr.dtype)
        nb_threads = self._NUM_WARPS * self._WARP_SIZE
        if True:
            # turn on google colaboraty
            if nb_threads > arr.size:
                nb_threads = arr.size
            nb_blocks = (arr.size + nb_threads - 1) // nb_threads
            temp = cuda.device_array(shape=nb_blocks, dtype=arr.dtype, stream=stream)
            kernel[nb_blocks, nb_threads, stream](arr, temp, null_value)

        # Perform the reduction on the GPU
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        while True:
            if nb_threads > arr.size:
                nb_threads = arr.size

            # TODO implements the logics

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        # handle return value
        if res is not None:
            res.copy_to_device(arr)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # Returned, not printed: the caller decides what the timing is
        # for. A benchmark keeps the number, a self-check ignores it,
        # and neither has to scrape it back out of the console.
        return arr.copy_to_host(stream=stream)[0], ct


#### Tests — exercise 4 — the commutative reduce

In [35]:
from __future__ import annotations

import numpy as np
from numba.np.numpy_support import from_dtype
from numba import cuda, core, config


class CommutativeReduce(object):
    """Create a reduction object that reduces values using a given binary
    function. The binary function is compiled once and cached inside this
    object. Keeping this object alive will prevent re-compilation.
    """

    _cache = {}

    _WARP_SIZE = 32
    _NUM_WARPS = 8

    @staticmethod
    def _gpu_kernel_factory(fn, np_type):
        """Factory of kernels for the reduction problem.

        This function returns a Cuda Kernel that does the reduction of some
        data using a given binary functor.
        """

        reduce_op = cuda.jit(device=True)(fn)

        max_block_size = (
            CommutativeReduce._NUM_WARPS
            * CommutativeReduce._WARP_SIZE
        )

        @cuda.jit(device=True)
        def load_shared_memory(shared_memory, arr, null_value):
            global_tid = cuda.grid(1)
            local_tid = cuda.threadIdx.x

            # Local reduction using a modulo strategy.
            acc = null_value
            i = global_tid

            while i < arr.size:
                acc = reduce_op(acc, arr[i])
                i += cuda.gridsize(1)

            shared_memory[local_tid] = acc

            # Wait for all threads.
            cuda.syncthreads()

        @cuda.jit(device=True)
        def reduce_per_warp(shared_memory):
            tid = cuda.threadIdx.x
            in_warp_id = tid % 32

            jump = 1
            while jump < 32:
                if in_warp_id % (2 * jump) == 0:
                    shared_memory[tid] = reduce_op(
                        shared_memory[tid],
                        shared_memory[tid + jump]
                    )

                cuda.syncthreads()
                jump *= 2

        def gpu_reduce_block(arr, partials, null_value):
            """
            Per block reduction.
            """
            shared_memory = cuda.shared.array(
                shape=max_block_size,
                dtype=np_type
            )

            # Move locally reduced values to shared memory.
            load_shared_memory(shared_memory, arr, null_value)

            # Reduce within each warp.
            reduce_per_warp(shared_memory)
            cuda.syncthreads()

            # Reduce the warp results.
            jump = 1
            while jump < 8:
                tid = cuda.threadIdx.x

                if tid < 8 and tid % (2 * jump) == 0:
                    shared_memory[tid * 32] = reduce_op(
                        shared_memory[tid * 32],
                        shared_memory[(tid + jump) * 32]
                    )

                cuda.syncthreads()
                jump *= 2

            # Store the result.
            if cuda.threadIdx.x == 0:
                blk_id = cuda.blockIdx.x
                partials[blk_id] = shared_memory[0]

        return cuda.jit(gpu_reduce_block)

    def __init__(self, functor):
        """
        :param functor: A function implementing a binary operation for
                        reduction. It will be compiled as a CUDA device
                        function using ``cuda.jit(device=True)``.
        """
        self._functor = functor

    def _compile(self, dtype):
        key = self._functor, dtype
        if key not in self._cache:
            self._cache[key] = CommutativeReduce._gpu_kernel_factory(
                self._functor, from_dtype(dtype)
            )
        return self._cache[key]

    def __call__(
        self,
        arr,
        null_value,
        res=None,
        stream=cuda.default_stream()
    ):
        """Performs a full reduction."""

        # Ensure 1D array.
        if arr.ndim != 1:
            raise TypeError("only support 1D array")

        # Ensure size > 0.
        if arr.size < 1:
            raise ValueError("array's length is 0")

        _sav, core.config.CUDA_LOW_OCCUPANCY_WARNINGS = (
            core.config.CUDA_LOW_OCCUPANCY_WARNINGS,
            False
        )

        kernel = self._compile(arr.dtype)
        nb_threads = self._NUM_WARPS * self._WARP_SIZE

        # Perform the reduction on the GPU.
        start_event = cuda.event(True)
        start_event.record(stream=stream)

        while True:
            nb_blocks = min(
                (arr.size + nb_threads - 1) // nb_threads,
                nb_threads
            )

            temp = cuda.device_array(
                shape=nb_blocks,
                dtype=arr.dtype,
                stream=stream
            )

            kernel[nb_blocks, nb_threads, stream](
                arr, temp, null_value
            )

            arr = temp

            if nb_blocks == 1:
                break

        stop_event = cuda.event(True)
        stop_event.record(stream=stream)
        stop_event.synchronize()
        ct = cuda.event_elapsed_time(start_event, stop_event)

        # Handle return value.
        if res is not None:
            res.copy_to_device(arr, stream=stream)

        core.config.CUDA_LOW_OCCUPANCY_WARNINGS = _sav

        # Return result and elapsed time.
        return arr.copy_to_host(stream=stream)[0], ct


def test4(h_input, reduce_class=CommutativeReduce):
    def check4(result):
        verify(
            f"sum of {h_input.size} values",
            result,
            h_input.dtype.type(h_input.sum())
        )

    reducer = reduce_class(lambda a, b: a + b)
    result, ct = reducer(
        cuda.to_device(h_input),
        h_input.dtype.type(0)
    )
    check4(result)
    return ct


rng = np.random.default_rng(4)

for size in (1 << 12, 1 << 18, (1 << 20) + 7):
    test4(
        rng.integers(
            0, 1 << 10,
            size=size,
            dtype=np.int32
        )
    )

ReduceExercise4 = CommutativeReduce

PASS  sum of 4096 values  (1 value)
PASS  sum of 262144 values  (1 value)
PASS  sum of 1048583 values  (1 value)


#### Benchmark — the commutative reduce

In [36]:
def build(size):
    h_input = np.random.default_rng(size).integers(
        0, 1 << 10, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    reducer = CommutativeReduce(lambda a, b: a + b)
    zero = np.int32(0)
    return ((lambda: reducer(d_input, zero)), (lambda: h_input.sum()),
            h_input.nbytes)


benchmark("commutative reduce: GPU against numpy.sum",
          build, (1 << 20, 1 << 22, 1 << 24))

commutative reduce: GPU against numpy.sum   —   this card peaks at 448 GB/s
        size |  GPU (ms) |  NumPy (ms) |  speed-up |    GB/s |  of peak
------------------------------------------------------------------------
     1048576 |     0.475 |       0.258 |      0.5x |     8.8 |     2.0%
     4194304 |     0.286 |       1.107 |      3.9x |    58.7 |    13.1%
    16777216 |     0.456 |       5.215 |     11.4x |   147.2 |    32.8%


## Afterwards — the four reduces, side by side

Each exercise added one idea to the previous one: the general reduce, the same
reduction holding less shared memory, one reduce per warp, then the modulo
scheme that hands every thread a slice of the array. The cell below puts all
four in one table, with the column that decides whether a reduce is good — how
much of the card's bandwidth it reaches.

The `kernel` column is your own measurement: every `__call__` here brackets its
reduction loop with CUDA events and returns `(value, ct)`. The `whole call`
column is wall-clock around the same call, so the distance between the two is
the price of everything the loop is not — allocations, launches, the copy back.
The four share one device copy of the input, so no PCIe transfer lands in any
measurement.

In [37]:
# --- provided: the four reduces of this labwork, measured --------------------
REDUCERS = [
    ("1 - the general reduce", ReduceExercise1),
    ("2 - less shared memory", ReduceExercise2),
    ("3 - one reduce per warp", ReduceExercise3),
    ("4 - the modulo scheme", ReduceExercise4),
]

def reduce_once(reducer, d_input, zero):
    """One reduction: the answer, and the kernel time the class measured.

    Every `__call__` of this labwork returns `(value, ct)`, so there is nothing
    to parse out of the console and no second stopwatch to disagree with. The
    redirect is only there in case your own class prints something: nothing in
    the provided code does any more.
    """
    with contextlib.redirect_stdout(io.StringIO()):
        return reducer(d_input, zero)


def compare(size):
    # Small values, so that the exact sum of 2^24 of them still fits in an
    # int32 and the check below can be an equality rather than a tolerance.
    h_input = np.random.default_rng(size).integers(0, 4, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    zero = np.int32(0)
    expected = np.int32(h_input.sum())

    # Nothing is timed before it is known to be right.
    reducers = [(label, cls(lambda a, b: a + b)) for label, cls in REDUCERS]
    for label, reducer in reducers:
        result, _ = reduce_once(reducer, d_input, zero)
        verify(f"reduce {label}", result, expected)
    print()

    peak = peak_bandwidth()
    print(f"REDUCE of 2^{size.bit_length() - 1} int32 ({h_input.nbytes / 1e6:.0f} MB)"
          + (f"   -   this card peaks at {peak:.0f} GB/s" if peak else ""))
    print(f"{'exercise':<26} | {'kernel (ms)':>11} | {'GB/s':>7} | "
          f"{'of peak':>8} | {'whole call (ms)':>15}")
    print("-" * 79)
    for label, reducer in reducers:
        repeats = 10
        kernel = min(reduce_once(reducer, d_input, zero)[1]
                     for _ in range(repeats))
        call = min(_time_once(lambda: reducer(d_input, zero))
                   for _ in range(repeats))
        rate = h_input.nbytes / (kernel / 1000.0) / 1e9
        share = "       -" if peak is None else f"{100 * rate / peak:7.1f}%"
        print(f"{label:<26} | {kernel:11.3f} | {rate:7.1f} | {share} | "
              f"{call:15.3f}")

    cpu = min(_time_once(lambda: h_input.sum()) for _ in range(3))
    print(f"{'numpy.sum, one core':<26} | {cpu:11.3f} | "
          f"{h_input.nbytes / (cpu / 1000.0) / 1e9:7.1f} | {'host':>8} | "
          f"{cpu:15.3f}")


# 2^20 shows what the call costs around the kernel; 2^26 is the size the
# cache-hint cell below uses, so the last row of this table and the first row
# of that one are the same kernel shape measured twice, independently.
for array_size in (1 << 20, 1 << 24, 1 << 26):
    compare(array_size)
    print()

PASS  reduce 1 - the general reduce  (1 value)
PASS  reduce 2 - less shared memory  (1 value)
PASS  reduce 3 - one reduce per warp  (1 value)
PASS  reduce 4 - the modulo scheme  (1 value)

REDUCE of 2^20 int32 (4 MB)   -   this card peaks at 448 GB/s
exercise                   | kernel (ms) |    GB/s |  of peak | whole call (ms)
-------------------------------------------------------------------------------
1 - the general reduce     |       0.226 |    18.6 |     4.1% |           0.426
2 - less shared memory     |       0.296 |    14.2 |     3.2% |           0.499
3 - one reduce per warp    |       0.258 |    16.3 |     3.6% |           0.362
4 - the modulo scheme      |       0.167 |    25.1 |     5.6% |           0.282
numpy.sum, one core        |       0.263 |    16.0 |     host |           0.263

PASS  reduce 1 - the general reduce  (1 value)
PASS  reduce 2 - less shared memory  (1 value)
PASS  reduce 3 - one reduce per warp  (1 value)
PASS  reduce 4 - the modulo scheme  (1 value)


### What a T4 answered about the four

At 2^26 int32 — 268 MB, on a card that peaks at 320 GB/s:

| exercise | kernel (ms) | GB/s | of peak | whole call (ms) |
|---|---|---|---|---|
| 1 — the general reduce | 10.245 | 26.2 | 8.2 % | 12.714 |
| 2 — less shared memory | 3.674 | 73.1 | 22.8 % | 8.302 |
| 3 — one reduce per warp | 2.806 | 95.7 | 29.9 % | 6.126 |
| 4 — the modulo scheme | **1.139** | **235.7** | **73.6 %** | 2.410 |
| `numpy.sum`, one core | 39.471 | 6.8 | *host* | 39.471 |

**Exercise 4 is nearly finished.** 73.6 % of theoretical peak is close to the
ceiling a T4 reaches on a pure streaming read; most of the missing quarter is
not there to be taken. What matters is that exercise 1 was leaving 92 % on the
table, and three ordinary ideas recovered nearly all of it.

**Fixing the kernel promotes the overhead to being the problem.** Read the last
column against the first. For exercise 4 the allocations, launches and copy
back now cost *more than the reduction itself*: the kernel is 9 times faster
than exercise 1's, the call a user makes only 5.3 times faster.

**Quote the speed-up over the CPU carefully.** Exercise 4's kernel is 35 times
faster than `numpy.sum`; the whole call is 16 times faster, and that is the
honest figure. One core is also not *the CPU*: a threaded sum would close part
of the gap.

Finally, compare the last row here with the first row of the next table: the
same kernel at the same size, measured by two cells sharing no code, 1.139 ms
against 1.186 ms. That 4 % is this notebook's measurement noise.

## Afterwards, second part — a branch that was never a branch

Exercise 4's inner loop combines two values the obvious way:

```python
if value > total:
    total = value
```

A warp issues one instruction for all thirty-two of its lanes, so when some
lanes take a branch and others do not, the warp runs both sides in turn with
the idle lanes switched off. That is divergence, and this condition depends on
the data, so the lanes should split roughly evenly on every element.

The GPU has an instruction for choosing a value without branching: `selp`,
which numba exposes as `cuda.selp(condition, a, b)` — `a` when the condition
holds, `b` otherwise. Replacing the `if` with it should remove the divergence
and win.

The cell below is exercise 4's kernel with that one line swapped and nothing
else, so whatever the table shows belongs to the combine step. The running
maximum lives in a *register*, which is what makes this the right kernel for
the question: a conditional assignment to shared memory would drag in a store.

**Read the `spread` column before reading the rest of the table.** The same
comparison on labwork 1's maximum gave three different verdicts at three sizes,
from byte-identical PTX — because a 1024-thread block fills a Turing SM by
itself, so every `syncthreads` stalls it with no other block to cover it, and
the timings reported whatever the clocks were doing. When one variant moves by
2.3x between runs, a 1.7x difference between two variants is not a result.

In [38]:
import re

from numba import int32

if not hasattr(cuda, "selp"):
    print(f"This numba ({numba.__version__}) does not expose cuda.selp, so this")
    print("section cannot run. Nothing above it is affected.")
else:
    THREADS = 256
    BLOCKS = 1024
    SIZES = (1 << 26, 1 << 28)

    def build_max(branchless):
        """Exercise 4's shape, maximum instead of sum, one line different.

        `branchless` is a closure variable, so numba folds it away when it
        compiles: each kernel contains one of the two combine steps and no
        trace of the other. Everything below the loop is identical in both.
        """
        @cuda.jit
        def kernel(d_input, d_partial):
            shared = cuda.shared.array(shape=THREADS, dtype=int32)
            tid = cuda.threadIdx.x

            # The modulo scheme: a private running maximum in a register,
            # walking the array with a stride of the whole grid. `d_input[0]`
            # is the starting value rather than zero -- it is a real element of
            # the array, so it is safe even when every value is negative, and a
            # thread with nothing to do simply keeps it.
            total = d_input[0]
            index = cuda.grid(1)
            step = cuda.gridsize(1)
            while index < d_input.size:
                value = d_input[index]
                if branchless:
                    total = cuda.selp(value > total, value, total)
                else:
                    if value > total:
                        total = value
                index += step

            # From here down the two kernels are the same, deliberately: the
            # tree runs log2(THREADS) times and the loop above runs once per
            # element, so this part is not where the comparison lives.
            shared[tid] = total
            cuda.syncthreads()

            jump = THREADS // 2
            while jump > 0:
                if tid < jump:
                    other = shared[tid + jump]
                    if other > shared[tid]:
                        shared[tid] = other
                cuda.syncthreads()
                jump //= 2

            if tid == 0:
                d_partial[cuda.blockIdx.x] = shared[0]

        return kernel

    def measure_paired(kernels, d_input, d_partial, repeats=20):
        """Time the variants interleaved, and keep every sample.

        Timing all of A and then all of B is the natural way to write this and
        the wrong way to run it. Twenty launches in a row warm the card, and a
        T4 answers a warm question more slowly than a cold one, so whichever
        variant went second would carry a penalty that has nothing to do with
        its code. Alternating puts both under the same conditions.

        Keeping every sample rather than only the best matters just as much.
        The best of twenty says what the card can do; the spread says whether
        it held still long enough for the comparison to mean anything, and on a
        machine you share that is the question to answer first.
        """
        samples = {label: [] for label, _ in kernels}
        for _ in range(repeats):
            for label, kernel in kernels:
                start, stop = cuda.event(True), cuda.event(True)
                start.record()
                kernel[BLOCKS, THREADS](d_input, d_partial)
                stop.record()
                stop.synchronize()
                samples[label].append(cuda.event_elapsed_time(start, stop))
        return samples

    def median(values):
        ordered = sorted(values)
        middle = len(ordered) // 2
        return (ordered[middle] if len(ordered) % 2
                else (ordered[middle - 1] + ordered[middle]) / 2)

    def instructions(kernel):
        """Count the instructions of interest in the PTX numba generated.

        This is the evidence for everything claimed above: rather than assert
        what the compiler does with an `if`, ask it. `inspect_asm()` returns
        the PTX of each compiled signature, and the kernel has been launched
        once by the time this runs, so there is one.
        """
        try:
            ptx = next(iter(kernel.inspect_asm().values()))
        except Exception:
            return None

        def count(pattern):
            # `[.]` rather than an escaped dot: this cell lives inside a
            # non-raw string in the generator, where a backslash is eaten.
            return len(re.findall(pattern, ptx))

        return {
            "max.s32": count("max[.]s32"),
            "setp.": count("setp[.]"),
            "selp.": count("selp[.]"),
            "bra": count("bra[.; ]"),
            "ld.global": count("ld[.]global"),
            "st.shared": count("st[.]shared"),
        }

    variants = [("a branch (if)", False), ("cuda.selp", True)]
    built = [(label, build_max(branchless)) for label, branchless in variants]
    peak = peak_bandwidth()

    print(f"MAXIMUM of negative int32, {BLOCKS} blocks x {THREADS} threads"
          + (f"   -   this card peaks at {peak:.0f} GB/s" if peak else ""))

    checked = False
    for size in SIZES:
        # Negative values on purpose: zero is not a safe starting point for a
        # maximum over them, and this is the array that catches it.
        h_input = np.random.default_rng(0).integers(
            -(1 << 30), -1, size=size, dtype=np.int32)
        d_input = cuda.to_device(h_input)
        d_partial = cuda.device_array(shape=BLOCKS, dtype=np.int32)

        for label, kernel in built:
            kernel[BLOCKS, THREADS](d_input, d_partial)   # warm-up: pays the JIT
            cuda.synchronize()
            if not checked:
                # The last BLOCKS values are reduced on the host: 1024 numbers
                # next to 67 million, and a second kernel would only blur what
                # this table compares.
                verify(f"maximum, {label}",
                       d_partial.copy_to_host().max(), h_input.max())

        samples = measure_paired(built, d_input, d_partial)
        if not checked:
            checked = True
            print()
            print(f"{'combine step':<22} | {'size':>5} | {'best':>7} | "
                  f"{'median':>7} | {'GB/s':>7} | {'of peak':>8} | {'spread':>7}")
            print("-" * 82)

        worst_spread, middles = 0.0, {}
        for label, _ in built:
            values = samples[label]
            best, mid = min(values), median(values)
            middles[label] = mid
            spread = max(values) / best
            worst_spread = max(worst_spread, spread)
            rate = h_input.nbytes / (mid / 1000.0) / 1e9
            share = "       -" if peak is None else f"{100 * rate / peak:7.1f}%"
            print(f"{label:<22} | 2^{size.bit_length() - 1:<3} | {best:7.3f} | "
                  f"{mid:7.3f} | {rate:7.1f} | {share} | {spread:6.2f}x")

        # A comparison is only worth reading when it is bigger than the
        # movement of the card underneath it.
        gap = max(middles.values()) / min(middles.values())
        slower = max(middles, key=middles.get)
        if gap < worst_spread:
            print(f"{'':<22} |    -> gap {gap:.2f}x, inside this row's own "
                  f"{worst_spread:.2f}x spread: no verdict")
        else:
            print(f"{'':<22} |    -> {slower} is {gap:.2f}x slower, against a "
                  f"{worst_spread:.2f}x spread")

    counted = [(label, instructions(kernel)) for label, kernel in built]
    if all(counts for _, counts in counted):
        print()
        print("and this is what the compiler emitted for each of them:")
        columns = list(counted[0][1])
        print(f"{'combine step':<22} | " +
              " | ".join(f"{op:>10}" for op in columns))
        print("-" * (24 + 13 * len(columns)))
        for label, counts in counted:
            print(f"{label:<22} | " +
                  " | ".join(f"{counts[op]:10d}" for op in columns))
        print()
        print("`max.s32` is the column to read first. On labwork 1's kernel it")
        print("appeared in the `cuda.selp` row and not in the `if` row: the")
        print("compiler recognised `select(a > b, a, b)` as a maximum and used")
        print("the single hardware instruction for it, while leaving the `if`")
        print("to pay an explicit compare. If that repeats here, then writing")
        print("`selp` by hand did not hide a pattern from the compiler -- it")
        print("handed it one -- and whatever the timing says, it is not saying")
        print("that one of these does less arithmetic than the other.")
        print()
        print("`bra` is the premise of the section, worth checking rather than")
        print("believing: a conditional assignment this small should be")
        print("predicated rather than jumped around, so neither kernel should")
        print("show a branch per element.")

MAXIMUM of negative int32, 1024 blocks x 256 threads   -   this card peaks at 448 GB/s
PASS  maximum, a branch (if)  (1 value)
PASS  maximum, cuda.selp  (1 value)

combine step           |  size |    best |  median |    GB/s |  of peak |  spread
----------------------------------------------------------------------------------
a branch (if)          | 2^26  |   0.635 |   0.658 |   408.0 |    91.1% |   1.09x
cuda.selp              | 2^26  |   0.635 |   0.660 |   406.9 |    90.8% |   1.18x
                       |    -> gap 1.00x, inside this row's own 1.18x spread: no verdict
a branch (if)          | 2^28  |   2.517 |   5.006 |   214.5 |    47.9% |   2.05x
cuda.selp              | 2^28  |   2.517 |   3.777 |   284.3 |    63.4% |   2.02x
                       |    -> gap 1.33x, inside this row's own 2.05x spread: no verdict

and this is what the compiler emitted for each of them:
combine step           |    max.s32 |      setp. |      selp. |        bra |  ld.global |  st.shared
-------

## Afterwards, third part — telling the cache what you intend

The card has a cache hierarchy, and so far nobody has told it how this array
will be used. Its guess is tuned for code that reads a value and reads it
again. A reduce is the opposite: every element is read **exactly once**, so
caching it is not merely useless but harmful, since each line evicts one
something else might want. numba exposes the instructions for saying so
([reference](https://nvidia.github.io/numba-cuda/reference/cache_hints.html)):

| instead of `a[i]` | policy | what you are promising |
|---|---|---|
| `cuda.ldca(a, i)` | cache-all | the default: keep it at every level |
| `cuda.ldcg(a, i)` | cache-global | skip L1, keep it in L2 |
| `cuda.ldcs(a, i)` | cache-streaming | it is passing through; evict it first |
| `cuda.ldlu(a, i)` | last-use | this is the last read; the line is free after |
| `cuda.ldcv(a, i)` | cache-volatile | do not trust the cache |

There are matching hints for stores — `stcg`, `stcs`, `stwb`, `stwt` — which
concern the partial sums rather than the input.

On paper `ldcs` and `ldlu` both describe this loop. Which one helps depends on
the card and on whether the kernel is short of bandwidth or of somewhere to put
what it has fetched, and a T4 answers differently from an A100. So the cell
below measures all five against plain indexing.

In [39]:
# --- provided: does a cache hint help a reduce? ------------------------------
from numba import int32

NEEDED = ("ldca", "ldcg", "ldcs", "ldlu", "ldcv")
missing = [name for name in NEEDED if not hasattr(cuda, name)]

if missing:
    # The version that decides this is numba-cuda's, not numba's: `numba.cuda`
    # is served by that separate distribution, and the two are numbered
    # independently.
    try:
        from importlib.metadata import version
        installed = f"numba-cuda {version('numba-cuda')}"
    except Exception:
        installed = f"numba {numba.__version__}, with no numba-cuda installed"

    print(f"{installed} does not expose {', '.join(missing)}.")
    print()
    print("The cache-hint intrinsics were added to numba-cuda in November 2025")
    print("and ship in 0.21.0 and later. If the line above says less than that,")
    print("upgrading *only* numba-cuda is enough -- it asks for numba >= 0.60,")
    print("so it will not drag numba itself forward:")
    print()
    print("    !pip install -q -U numba-cuda")
    print()
    print("Mind what else pins it, though: a Colab GPU image ships cudf, cuml")
    print("and dask-cuda, and those hold numba-cuda to a narrow range. Check")
    print("the version you already have before upgrading past them. Nothing")
    print("above this cell depends on any of it.")
else:
    THREADS = 256
    BLOCKS = 1024

    # One device function per policy, each written out rather than generated,
    # because the hint has to be named literally inside the compiled function.
    @cuda.jit(device=True)
    def load_plain(a, i):
        return a[i]

    @cuda.jit(device=True)
    def load_ca(a, i):
        return cuda.ldca(a, i)

    @cuda.jit(device=True)
    def load_cg(a, i):
        return cuda.ldcg(a, i)

    @cuda.jit(device=True)
    def load_cs(a, i):
        return cuda.ldcs(a, i)

    @cuda.jit(device=True)
    def load_lu(a, i):
        return cuda.ldlu(a, i)

    @cuda.jit(device=True)
    def load_cv(a, i):
        return cuda.ldcv(a, i)

    LOADS = [
        ("a[i]  (no hint given)", load_plain),
        ("cuda.ldca  cache-all", load_ca),
        ("cuda.ldcg  cache-global", load_cg),
        ("cuda.ldcs  cache-streaming", load_cs),
        ("cuda.ldlu  last-use", load_lu),
        ("cuda.ldcv  cache-volatile", load_cv),
    ]

    def build_reduce(load):
        """A sum whose only variable is how it reads global memory.

        The shape is exercise 4's: each thread walks the array with a stride of
        the whole grid, keeping a running total in a register, and only then do
        the threads of a block combine what they hold. That shape is the point
        -- it is the one where the loads dominate and a hint has room to matter.
        """
        @cuda.jit
        def kernel(d_input, d_partial):
            shared = cuda.shared.array(shape=THREADS, dtype=int32)
            tid = cuda.threadIdx.x

            total = int32(0)
            index = cuda.grid(1)
            step = cuda.gridsize(1)
            while index < d_input.size:
                total += load(d_input, index)
                index += step

            shared[tid] = total
            cuda.syncthreads()

            jump = THREADS // 2
            while jump > 0:
                if tid < jump:
                    shared[tid] += shared[tid + jump]
                cuda.syncthreads()
                jump //= 2

            if tid == 0:
                d_partial[cuda.blockIdx.x] = shared[0]

        return kernel

    def measure(kernel, d_input, d_partial, repeats=20):
        """Best of `repeats`, timed on the device by CUDA events."""
        best = float("inf")
        for _ in range(repeats):
            start, stop = cuda.event(True), cuda.event(True)
            start.record()
            kernel[BLOCKS, THREADS](d_input, d_partial)
            stop.record()
            stop.synchronize()
            best = min(best, cuda.event_elapsed_time(start, stop))
        return best

    # 2^26 int32 is 268 MB: far more than any cache on the card holds, which is
    # what makes the question about policy rather than about luck.
    size = 1 << 26
    h_input = np.random.default_rng(0).integers(0, 4, size=size, dtype=np.int32)
    d_input = cuda.to_device(h_input)
    d_partial = cuda.device_array(shape=BLOCKS, dtype=np.int32)
    expected = np.int32(h_input.sum())
    peak = peak_bandwidth()

    print(f"REDUCE of 2^{size.bit_length() - 1} int32 "
          f"({h_input.nbytes / 1e6:.0f} MB), {BLOCKS} blocks x {THREADS} threads"
          + (f"   -   this card peaks at {peak:.0f} GB/s" if peak else ""))
    print()

    built = []
    for label, load in LOADS:
        kernel = build_reduce(load)
        kernel[BLOCKS, THREADS](d_input, d_partial)     # warm-up: pays the JIT
        cuda.synchronize()
        # The last BLOCKS values are summed on the host: 1024 numbers next to
        # 67 million, and a second kernel would only blur the comparison.
        verify(f"sum with {label.split()[0]}",
               np.int32(d_partial.copy_to_host().sum()), expected)
        built.append((label, kernel))
    print()

    print(f"{'load':<28} | {'kernel (ms)':>11} | {'GB/s':>7} | {'of peak':>8}")
    print("-" * 62)
    baseline = None
    for label, kernel in built:
        ms = measure(kernel, d_input, d_partial)
        baseline = ms if baseline is None else baseline
        rate = h_input.nbytes / (ms / 1000.0) / 1e9
        share = "       -" if peak is None else f"{100 * rate / peak:7.1f}%"
        print(f"{label:<28} | {ms:11.3f} | {rate:7.1f} | {share}")
    print()
    print("If the spread here is smaller than the run-to-run variation, run the")
    print("cell again before drawing a conclusion from it.")

    # --- and what the compiler actually emitted -----------------------------
    # The timing says the hints cost something. This says what changed, which
    # is the only way to tell a cache-policy effect from a code-generation one.
    import re

    POLICIES = ("nc", "ca", "cg", "cs", "lu", "cv", "wb", "wt")

    def global_ops(kernel, opcode):
        """Count the forms of `ld.global` or `st.global` in a kernel's PTX.

        A qualifier such as `.ca`, `.cs` or `.wt` is the cache policy. `.nc` is
        the odd one out: it is the read-only path, which the compiler picks by
        itself when it can prove nothing writes the array, and which no
        intrinsic can ask for -- there is no store counterpart to it. The type
        suffix (`.u32`) is dropped, since it is not the question here.
        """
        try:
            ptx = next(iter(kernel.inspect_asm().values()))
        except Exception:
            return None
        counts = {}
        # `[.]` rather than an escaped dot: this whole cell lives inside a
        # non-raw string in the generator, where a backslash would be eaten.
        for match in re.finditer(opcode + "[.]global((?:[.][a-z0-9]+)*)", ptx):
            policy = [part for part in match.group(1).split(".")
                      if part in POLICIES]
            key = f"{opcode}.global." + ".".join(policy) if policy                 else f"{opcode}.global"
            counts[key] = counts.get(key, 0) + 1
        return counts

    for opcode, what in (("ld", "reads"), ("st", "writes")):
        print()
        print(f"{'load':<28} | how the PTX {what} global memory")
        print("-" * 62)
        for label, kernel in built:
            counts = global_ops(kernel, opcode)
            if counts is None:
                print(f"{label:<28} | (PTX not available on this numba)")
                continue
            print(f"{label:<28} | "
                  + ", ".join(f"{k} x{v}" for k, v in sorted(counts.items())))

    print()
    print("Read the first table against the timings. If the unhinted kernel is")
    print("the only one using `ld.global.nc`, the hints did not lose on cache")
    print("policy -- they lost by taking the read-only path away from the")
    print("compiler.")
    print()
    print("The second table is the same question asked about the writes, and")
    print("it should look far duller: every row writes the same way, because")
    print("nothing in this comparison ever varied the store. That is the point")
    print("of printing it. The store policy is not something the compiler")
    print("out-thinks you on the way `ld.global.nc` is -- it is simply not")
    print("worth deciding here, since these kernels write one value per block:")
    print(f"{BLOCKS * 4} bytes out of the {h_input.nbytes // 1000000} MB they read.")

REDUCE of 2^26 int32 (268 MB), 1024 blocks x 256 threads   -   this card peaks at 448 GB/s

PASS  sum with a[i]  (1 value)
PASS  sum with cuda.ldca  (1 value)
PASS  sum with cuda.ldcg  (1 value)
PASS  sum with cuda.ldcs  (1 value)
PASS  sum with cuda.ldlu  (1 value)
PASS  sum with cuda.ldcv  (1 value)

load                         | kernel (ms) |    GB/s |  of peak
--------------------------------------------------------------
a[i]  (no hint given)        |       0.637 |   421.1 |    94.0%
cuda.ldca  cache-all         |       0.657 |   408.4 |    91.2%
cuda.ldcg  cache-global      |       0.650 |   413.2 |    92.2%
cuda.ldcs  cache-streaming   |       0.622 |   431.6 |    96.3%
cuda.ldlu  last-use          |       0.659 |   407.6 |    91.0%
cuda.ldcv  cache-volatile    |       0.657 |   408.8 |    91.2%

If the spread here is smaller than the run-to-run variation, run the
cell again before drawing a conclusion from it.

load                         | how the PTX reads global memory
---

### What a T4 answered about the hints

Tesla T4, 2^26 int32, best of twenty runs:

| load | kernel (ms) | GB/s | of peak |
|---|---|---|---|
| `a[i]` — no hint given | **1.186** | **226.4** | **70.7 %** |
| `cuda.ldca` cache-all | 1.603 | 167.5 | 52.3 % |
| `cuda.ldcg` cache-global | 1.605 | 167.2 | 52.2 % |
| `cuda.ldcs` cache-streaming | 1.453 | 184.7 | 57.7 % |
| `cuda.ldlu` last-use | 1.436 | 187.0 | 58.4 % |
| `cuda.ldcv` cache-volatile | 1.595 | 168.3 | 52.6 % |

**Not one hint helps.** The unhinted load wins by 21 % over the best of them,
far outside the 4 % run-to-run variation established above.

The order among the five is not random, though: `ldcs` and `ldlu` — the two
that say *this data is passing through* — beat the other three by about 10 %.
Read-once really is what this loop does. They are simply not better than saying
nothing.

Hence the lesson: **a hint is worth giving only when you know something the
compiler cannot work out for itself.** Here it already sees that the reduce
never writes `d_input`, and it has a better instruction than any in the table —
the read-only path `ld.global.nc`, which no intrinsic can request. Reaching for
`cuda.ldcs` does not add a policy to that load, it *replaces* the load with an
inline-assembly one the optimiser cannot reorder. The PTX table below is where
you can watch that happen.

So the reduces and scans of the labworks that follow keep plain `a[i]`. One
caveat: this is a T4 answer, and a card with a much larger L2 could reverse it.

The stores are a different question, and this kernel cannot answer it: these
reduces write **one `int32` per block**, 4 KB against 268 MB read, so no store
policy can move a number that small. A pattern that writes as much as it reads
can, and measuring it there is the honest way to find out.